# UNI + BACH数据集 推理加速系统
---
**功能**: 加载UNI模型 → 提取BACH数据集特征 → 训练分类器 → 输出评估指标

**硬件**: 昇腾NPU 910B / NVIDIA GPU / CPU

---

## 1. 安装依赖

In [ ]:
# 安装必要的依赖包
!pip install torch==2.0.1 torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install timm==0.9.8 numpy pandas scikit-learn huggingface_hub opencv-python pillow joblib matplotlib tqdm
!pip install -e .

## 2. 导入库

In [ ]:
import os
import sys
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

from uni import get_encoder
from uni.optim import GraphOptimizer

print("=" * 50)
print("UNI + BACH Inference System")
print("=" * 50)

## 3. HuggingFace 登录 (首次运行必须)

**获取Token**: https://huggingface.co/settings/tokens

In [ ]:
# HuggingFace 登录
from huggingface_hub import login, hf_hub_download

# 如果已有token，取消下面的注释并填入
# login(token='your_token_here')

# 或手动输入token
print("请在下方输入HuggingFace Token:")
print("获取地址: https://huggingface.co/settings/tokens")
token = input("Token: ").strip()
if token:
    login(token=token)
    print("登录成功!")
else:
    print("跳过登录")

## 4. 配置参数 (⚠️ 请根据您的环境修改)

### 修改下面的配置后运行

In [ ]:
# ==================== 用户配置区 ====================
# 修改这里的配置即可运行
CONFIG = {
    # 模型配置
    'model_name': 'uni2-h',           # 'uni' 或 'uni2-h'
    'device': 'auto',                  # 'auto', 'npu:0', 'cuda:0', 'cpu'
    
    # 数据路径 (请修改为您的实际路径)
    'data_root': r'../Photos',
    'csv_file': 'microscopy_ground_truth.csv',
    
    # 优化选项
    'use_optimization': True,          # 启用NPU优化
    'batch_size': 16,                 # 批处理大小
    'use_amp': True,                  # 启用混合精度
    
    # 训练配置
    'train_ratio': 0.7,               # 训练集比例
    'val_ratio': 0.15,                # 验证集比例
    'random_seed': 42,                 # 随机种子
    
    # 输出
    'output_dir': './output',
    'save_features': True,
    'save_predictions': True,
    'save_classifier': True,
}
# =================================================

print("配置信息:")
print(f"  模型: {CONFIG['model_name']}")
print(f"  设备: {CONFIG['device']}")
print(f"  数据路径: {CONFIG['data_root']}")
print(f"  NPU优化: {'启用' if CONFIG['use_optimization'] else '禁用'}")

## 5. 加载UNI模型

In [ ]:
# 自动检测设备
if CONFIG['device'] == 'auto':
    if hasattr(torch, 'npu') and torch.npu.is_available():
        device = torch.device('npu:0')
    elif torch.cuda.is_available():
        device = torch.device('cuda:0')
    else:
        device = torch.device('cpu')
else:
    device = torch.device(CONFIG['device'])

print(f"使用设备: {device}")

# 加载UNI模型
print("加载UNI模型...")
model, transform = get_encoder(
    enc_name=CONFIG['model_name'],
    device=device,
    test_batch=True
)
print(f"✓ 模型加载成功: {CONFIG['model_name']}")

## 6. 应用NPU优化 (可选)

In [ ]:
if CONFIG['use_optimization']:
    print("应用NPU优化...")
    optimizer = GraphOptimizer(model, str(device))
    model = optimizer.optimize_for_inference(
        use_amp=CONFIG['use_amp'],
        enable_graph_compile=True
    )
    print("✓ 优化已应用 (整图下沉 + 混合精度)")
else:
    print("优化已禁用")

## 7. 加载BACH数据集

In [ ]:
# 读取CSV标签文件
csv_path = os.path.join(CONFIG['data_root'], CONFIG['csv_file'])
df = pd.read_csv(csv_path, header=None, names=['filename', 'label'])

# 构建完整路径
df['filepath'] = df['label'] + '/' + df['filename']
df['full_path'] = df['filepath'].apply(lambda x: os.path.join(CONFIG['data_root'], x))

# 标签编码
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])

print(f"数据集大小: {len(df)} 张图像")
print(f"类别: {list(label_encoder.classes_)}")
print(f"\n类别分布:")
for label in label_encoder.classes_:
    count = (df['label'] == label).sum()
    print(f"  - {label}: {count} 张")

## 8. 提取特征向量

In [ ]:
def extract_features(model, df, transform, device, batch_size=16):
    """批量提取图像特征"""
    model.eval()
    features_list = []
    labels_list = []
    filenames_list = []
    
    with torch.no_grad():
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="提取特征"):
            try:
                image = Image.open(row['full_path']).convert('RGB')
                image_tensor = transform(image).unsqueeze(0).to(device)
                
                feat = model(image_tensor)
                
                # 处理输出维度
                if isinstance(feat, tuple):
                    feat = feat[0]
                if feat.dim() > 2:
                    feat = feat.mean(dim=1) if feat.dim() == 3 else feat.squeeze()
                
                features_list.append(feat.cpu().numpy())
                labels_list.append(row['label_encoded'])
                filenames_list.append(row['filename'])
                
            except Exception as e:
                print(f"\n警告: 处理 {row['filename']} 失败: {e}")
                features_list.append(np.zeros(1536))
                labels_list.append(row['label_encoded'])
                filenames_list.append(row['filename'])
    
    return np.array(features_list), np.array(labels_list), filenames_list

# 提取特征
print("开始提取特征向量...")
start_time = time.time()
features, labels, filenames = extract_features(
    model, df, transform, device, CONFIG['batch_size']
)
extract_time = time.time() - start_time

print(f"\n✓ 特征提取完成!")
print(f"  特征维度: {features.shape}")
print(f"  提取时间: {extract_time:.2f}秒")
print(f"  平均每张: {extract_time/len(df)*1000:.2f}ms")

## 9. 划分数据集

In [ ]:
# 计算划分比例
train_ratio = CONFIG['train_ratio']
val_ratio = CONFIG['val_ratio']
test_ratio = 1.0 - train_ratio - val_ratio

# 第一次划分: 分出测试集
X_temp, X_test, y_temp, y_test = train_test_split(
    features, labels,
    test_size=test_ratio,
    random_state=CONFIG['random_seed'],
    stratify=labels
)

# 第二次划分: 从剩余数据中分出验证集
val_size = val_ratio / (train_ratio + val_ratio)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=val_size,
    random_state=CONFIG['random_seed'],
    stratify=y_temp
)

print("数据集划分:")
print(f"  训练集: {len(X_train)} 张 ({len(X_train)/len(features)*100:.1f}%)")
print(f"  验证集: {len(X_val)} 张 ({len(X_val)/len(features)*100:.1f}%)")
print(f"  测试集: {len(X_test)} 张 ({len(X_test)/len(features)*100:.1f}%)")

## 10. 训练分类器

In [ ]:
# 训练Logistic Regression分类器
print("训练Logistic Regression分类器...")

clf = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver='lbfgs',
    multi_class='multinomial',
    random_state=CONFIG['random_seed'],
    n_jobs=-1
)

train_start = time.time()
clf.fit(X_train, y_train)
train_time = time.time() - train_start

# 计算准确率
train_acc = clf.score(X_train, y_train)
val_acc = clf.score(X_val, y_val)
test_acc = clf.score(X_test, y_test)

print(f"\n✓ 训练完成!")
print(f"  训练时间: {train_time:.2f}秒")
print(f"  训练集准确率: {train_acc:.4f}")
print(f"  验证集准确率: {val_acc:.4f}")
print(f"  测试集准确率: {test_acc:.4f}")

## 11. 评估性能

In [ ]:
# 预测
y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)

# 计算指标
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

# 分类报告
print("=" * 50)
print("测试集评估结果")
print("=" * 50)
print(f"\nAccuracy: {accuracy:.4f}")
print(f"F1-Score (macro): {f1_macro:.4f}")
print(f"F1-Score (weighted): {f1_weighted:.4f}")

print("\n详细分类报告:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

## 12. 混淆矩阵可视化

In [ ]:
# 绘制混淆矩阵
fig, ax = plt.subplots(figsize=(8, 6))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_
)

disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('BACH Dataset - Confusion Matrix\nUNI2-h + Logistic Regression', fontsize=12)
plt.tight_layout()
plt.show()

## 13. 保存结果

In [ ]:
# 创建输出目录
os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(os.path.join(CONFIG['output_dir'], 'models'), exist_ok=True)

# 保存特征向量
if CONFIG['save_features']:
    np.save(os.path.join(CONFIG['output_dir'], 'features.npy'), features)
    np.save(os.path.join(CONFIG['output_dir'], 'labels.npy'), labels)
    print("✓ 特征向量已保存: output/features.npy")

# 保存预测结果
if CONFIG['save_predictions']:
    pred_df = pd.DataFrame({
        'filename': filenames,
        'true_label': label_encoder.inverse_transform(labels),
        'pred_label': clf.predict(features).tolist(),
    })
    pred_df['pred_label'] = label_encoder.inverse_transform(pred_df['pred_label'])
    pred_df.to_csv(os.path.join(CONFIG['output_dir'], 'predictions.csv'), index=False)
    print("✓ 预测结果已保存: output/predictions.csv")

# 保存分类器模型
if CONFIG['save_classifier']:
    import joblib
    model_data = {
        'classifier': clf,
        'label_encoder': label_encoder,
        'config': CONFIG,
        'feature_dim': features.shape[1]
    }
    joblib.dump(model_data, os.path.join(CONFIG['output_dir'], 'models', 'classifier_model.pkl'))
    print("✓ 分类器模型已保存: output/models/classifier_model.pkl")

# 保存评估指标
metrics = {
    'accuracy': float(accuracy),
    'f1_macro': float(f1_macro),
    'f1_weighted': float(f1_weighted),
    'train_acc': float(train_acc),
    'val_acc': float(val_acc),
    'test_acc': float(test_acc),
    'confusion_matrix': cm.tolist(),
    'class_names': list(label_encoder.classes_),
    'feature_dim': features.shape[1],
    'num_samples': len(features)
}
with open(os.path.join(CONFIG['output_dir'], 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print("✓ 评估指标已保存: output/metrics.json")

# 保存混淆矩阵图
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('BACH Dataset - Confusion Matrix')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
print("✓ 混淆矩阵已保存: output/confusion_matrix.png")

## 14. 加载模型预测新图像

使用已训练的分类器对新的病理图像进行预测

In [ ]:
import joblib

def load_trained_system(model_path='./output/models/classifier_model.pkl'):
    """加载已训练的系统"""
    data = joblib.load(model_path)
    
    # 加载分类器
    clf = data['classifier']
    label_encoder = data['label_encoder']
    config = data['config']
    
    # 重新加载UNI模型
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    uni_model, transform = get_encoder(
        enc_name=config['model_name'],
        device=device,
        test_batch=True
    )
    
    return {
        'classifier': clf,
        'label_encoder': label_encoder,
        'uni_model': uni_model,
        'transform': transform,
        'device': device
    }

def predict_image(system, image_path):
    """预测单张图像"""
    clf = system['classifier']
    label_encoder = system['label_encoder']
    uni_model = system['uni_model']
    transform = system['transform']
    device = system['device']
    
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        feat = uni_model(image_tensor)
        if isinstance(feat, tuple):
            feat = feat[0]
        if feat.dim() > 2:
            feat = feat.mean(dim=1) if feat.dim() == 3 else feat.squeeze()
        feat = feat.cpu().numpy().reshape(1, -1)
    
    pred_label = clf.predict(feat)[0]
    pred_proba = clf.predict_proba(feat)[0]
    
    label_name = label_encoder.inverse_transform([pred_label])[0]
    confidence = pred_proba[pred_label]
    
    return {
        'label': label_name,
        'confidence': float(confidence),
        'probabilities': {cls: float(prob) for cls, prob in zip(label_encoder.classes_, pred_proba)}
    }

print("系统加载函数已定义")
print("\n使用方法:")
print("  system = load_trained_system()")
print("  result = predict_image(system, 'path/to/image.tif')")
print("  print(result['label'], result['confidence'])")

## 15. 示例: 预测新图像

In [ ]:
# 示例: 预测一张图像
# 请修改为您的图像路径

# 加载系统
system = load_trained_system()
print("✓ 系统加载成功")

# 预测示例 (请修改为实际图像路径)
test_image_path = os.path.join(CONFIG['data_root'], 'Normal/n001.tif')  # 示例路径

if os.path.exists(test_image_path):
    result = predict_image(system, test_image_path)
    
    print("\n" + "=" * 40)
    print("  预测结果")
    print("=" * 40)
    print(f"  预测类别: {result['label']}")
    print(f"  置信度: {result['confidence']:.2%}")
    print("\n  各类别概率:")
    for cls, prob in result['probabilities'].items():
        bar = '█' * int(prob * 20)
        print(f"    {cls:15s} {prob:.2%} {bar}")
    print("=" * 40)
else:
    print(f"图像不存在: {test_image_path}")
    print("请修改 test_image_path 为您的实际图像路径")

---

## 运行完成!

### 输出文件
| 文件 | 说明 |
|------|------|
| `output/features.npy` | 特征向量 |
| `output/predictions.csv` | 预测结果 |
| `output/metrics.json` | 评估指标 |
| `output/models/classifier_model.pkl` | 训练好的分类器 |
| `output/confusion_matrix.png` | 混淆矩阵图 |

### 下一步
1. 查看 `output/` 目录下的结果文件
2. 使用 `load_trained_system()` 加载模型预测新图像
3. 如需重新训练，修改配置后重新运行